# Part D. Model Evaluation, Visualization and Interpretation

Run the cells in order using the dependencies in `requirements.txt`. Start the kernel in the project root or the Part D directory. Outputs are written to the Part D directory.

## Imports and paths

In [ ]:
"""Run with Python, pandas, NumPy, scikit-learn, matplotlib, XGBoost and python-docx."""
from ast import literal_eval
from pathlib import Path
from textwrap import fill

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Inches
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import DMatrix, XGBClassifier

OUT = Path.cwd().resolve()
if not (OUT / "evaluate.py").is_file():
    OUT = OUT / "Model Evaluation Visualization and Interpretation"
ROOT = OUT.parent
SEED = 42
THREADS = 4
NAMES = ("Logistic Regression", "Random Forest", "XGBoost")
SETS = ("Base", "Base + ACE")
METRICS = {
    "Accuracy": accuracy_score, "Precision": precision_score,
    "Recall": recall_score, "F1": f1_score, "ROC-AUC": roc_auc_score,
}

## Data and model pipelines

In [ ]:
def load_inputs():
    data = pd.read_csv(ROOT / "Preparation and Exploratory Analysis" / "BRFSS_2023_ML_Feature_Dataset.csv")
    splits = pd.read_csv(ROOT / "analysisB" / "B_split_assignments.csv")
    if not np.array_equal(splits.row_index, data.index) or not splits.Dataset_Split.equals(data.Dataset_Split):
        raise ValueError("The modeling data and Part B split assignments differ.")
    if set(data.Dataset_Split) != {"train", "test"}:
        raise ValueError("Expected train and test assignments.")
    settings = {}
    cv = pd.read_csv(ROOT / "analysisC" / "analysisC" / "Part_C_results" / "cv_results.csv")
    for _, row in cv.iterrows():
        settings[row.Model, row.Features] = {
            key: literal_eval(value) if key != "solver" else value
            for key, value in (item.split("=") for item in row["Selected settings"].split(", "))
        }
    b_settings = pd.read_csv(ROOT / "analysisB" / "B_best_params.csv")
    for _, row in b_settings.iterrows():
        settings[row.Model, SETS[1]] = {
            key.removeprefix("clf__"): value for key, value in literal_eval(row.Best_Params).items()
            if key != "clf__penalty"
        }
    return data, settings


def make_pipeline(name, features, settings):
    categories = ["_RACEGR3", "_STATE", "EDUCA", "INCOME3", "MARITAL",
                  "_BMI5CAT", "DIABETE4", "_SMOKER3"]
    age = ["_AGE80"]
    binary = [feature for feature in features if feature not in categories + age]
    prep = ColumnTransformer([
        ("bin", SimpleImputer(strategy="most_frequent"), binary),
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), age),
        ("cat", Pipeline([
            ("fill", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=name != "XGBoost")),
        ]), categories),
    ])
    if name == NAMES[0]:
        model = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED, **settings)
    elif name == NAMES[1]:
        model = RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=1, **settings)
    else:
        model = XGBClassifier(n_estimators=160, objective="binary:logistic", tree_method="hist",
                              eval_metric="logloss", random_state=SEED, n_jobs=THREADS, **settings)
    return Pipeline([("prep", prep), ("clf", model)])

## Bootstrap intervals and figure export

In [ ]:
def bootstrap_changes(y, pairs):
    rng = np.random.default_rng(SEED)
    strata = [np.flatnonzero(y == label) for label in (0, 1)]
    changes = {name: [] for name in NAMES}
    for _ in range(1000):
        idx = np.concatenate([rng.choice(group, len(group), replace=True) for group in strata])
        yy = y[idx]
        for name, (base, ace) in pairs.items():
            changes[name].append([
                f1_score(yy, ace[0][idx]) - f1_score(yy, base[0][idx]),
                roc_auc_score(yy, ace[1][idx]) - roc_auc_score(yy, base[1][idx]),
            ])
    return {name: np.quantile(values, [0.025, 0.975], axis=0) for name, values in changes.items()}


def save_figure(fig, name):
    fig.savefig(OUT / name, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)

## Feature importance and TreeSHAP

In [ ]:
def feature_labels():
    dictionary = pd.read_csv(ROOT / "Preparation and Exploratory Analysis" / "Feature_Dictionary.csv")
    return dictionary.set_index("Variable").Description.to_dict()


def plot_importance(frame, labels, mean, error, xlabel, filename, title):
    model_count = frame.Model.nunique()
    size = (8, 4.6) if model_count == 1 else (5.2 * model_count, 4.0)
    fig, axes = plt.subplots(1, model_count, figsize=size, squeeze=False,
                             layout="constrained")
    for ax, (name, scores) in zip(axes.ravel(), frame.groupby("Model", sort=False)):
        top = scores.nlargest(9, mean).sort_values(mean)
        tick_labels = top.Feature.map(labels)
        if model_count == 1:
            tick_labels = tick_labels.map(lambda label: fill(label, width=32))
        ax.barh(tick_labels, top[mean], xerr=top[error] if error else None,
                color=["#d17838" if feature.startswith("ACE") else "#2d728f" for feature in top.Feature])
        ax.axvline(0, color="0.4", linewidth=0.8)
        ax.set(title=name, xlabel=xlabel)
    fig.suptitle(title)
    save_figure(fig, filename)


def xgb_shap(model, x, features):
    prep = model["prep"]
    transformed = prep.transform(x[features])
    booster = model["clf"].get_booster()
    values = booster.predict(DMatrix(transformed), pred_contribs=True)
    margins = booster.predict(DMatrix(transformed), output_margin=True)
    np.testing.assert_allclose(values.sum(axis=1), margins, atol=2e-5)
    contributions = values[:, :-1]
    categories = prep.named_transformers_["cat"]["encode"].categories_
    groups = list(prep.transformers_[0][2]) + list(prep.transformers_[1][2])
    for feature, levels in zip(prep.transformers_[2][2], categories):
        groups.extend([feature] * len(levels))
    group_names = np.array(groups)
    return pd.DataFrame({
        "Model": "XGBoost", "Feature": features,
        "Mean absolute SHAP": [
            np.abs(contributions[:, group_names == feature].sum(axis=1)).mean()
            for feature in features
        ],
    })

## Word report generation

In [ ]:
def add_table(doc, frame):
    table = doc.add_table(rows=1, cols=len(frame.columns))
    table.style = "Light Shading Accent 1"
    for cell, heading in zip(table.rows[0].cells, frame.columns):
        cell.text = heading
    for row in frame.itertuples(index=False, name=None):
        for cell, value in zip(table.add_row().cells, row):
            cell.text = f"{value:.4f}" if isinstance(value, float) else str(value)


def report(results, delta, importance, shap, labels, adherence):
    doc = Document()
    doc.add_heading("Part D. Model Evaluation, Visualization and Interpretation", 0)
    doc.add_paragraph(
        "We evaluated whether 11 adverse childhood experience (ACE) items add predictive information "
        "for current self-reported subjective cognitive decline (SCD) beyond 14 base predictors."
    )
    doc.add_heading("Holdout design", level=1)
    doc.add_paragraph(
        "All six pipelines used the shared split: 28,603 training respondents and 7,151 test respondents "
        "(1,223 SCD-positive). Imputation, age scaling and one-hot encoding were fitted on training rows. "
        "We fixed settings before scoring the test set, using the Part B parameter CSV for the two full "
        "B models and the Part C CV CSV for the remaining configurations. The Part C Word report gives "
        "a different base random-forest configuration, so this analysis identifies the CSV as its source. "
        "Classification metrics use a probability cutoff of 0.5. The two B pipelines were refitted from their "
        "recorded settings in the evaluation environment."
    )
    doc.add_paragraph(adherence)
    archive = pd.read_csv(ROOT / "analysisB" / "B_model_performance.csv")
    doc.add_paragraph(
        f"The archived Part B F1 scores were {archive.F1.iloc[0]:.4f} for logistic regression and "
        f"{archive.F1.iloc[1]:.4f} for random forest. The paired comparisons below use the six refits "
        "in one environment (scikit-learn 1.2.2 and XGBoost 3.1.3)."
    )
    doc.add_heading("Test-set performance", level=1)
    add_table(doc, results[["Model", "Features", *METRICS]])
    expanded = results[results.Features == SETS[1]]
    best_auc = expanded.loc[expanded["ROC-AUC"].idxmax()]
    best_recall = expanded.loc[expanded.Recall.idxmax()]
    doc.add_paragraph(
        f"Among the ACE models, {best_auc.Model} had the highest ROC-AUC ({best_auc['ROC-AUC']:.4f}). "
        f"{best_recall.Model} had the highest recall ({best_recall.Recall:.4f})."
    )
    for name in NAMES:
        row = results[(results.Model == name) & (results.Features == SETS[1])].iloc[0]
        doc.add_paragraph(
            f"With ACE, {name} identified {row.TP:,} of 1,223 SCD cases and flagged "
            f"{row.FP:,} of 5,928 respondents without SCD."
        )
    doc.add_paragraph("The majority-class accuracy baseline is 82.90%. It identifies zero SCD cases.")
    doc.add_picture(str(OUT / "confusion_matrices.png"), width=Inches(6.4))
    doc.add_picture(str(OUT / "roc_curves.png"), width=Inches(6.4))
    doc.add_heading("Incremental value of ACE", level=1)
    add_table(doc, delta)
    doc.add_paragraph(
        "Changes are computed as Base + ACE minus Base. Intervals are 95% paired percentile bootstrap intervals "
        "from 1,000 test-set resamples stratified by observed outcome. Each resample uses the same respondents "
        "for both feature sets. The intervals condition on the trained models and this cohort."
    )
    for metric, measure, interval in [
        ("F1", "F1 change", "F1 95% CI"),
        ("ROC-AUC", "AUC change", "AUC 95% CI"),
    ]:
        improved = delta.loc[delta[measure] > 0, "Model"].tolist()
        supported = [
            name for name, ci in zip(delta.Model, delta[interval])
            if float(ci.split(",")[0][1:]) > 0
        ]
        doc.add_paragraph(
            f"{metric} increased for {', '.join(improved)}. " if improved else
            f"No model had a higher test {metric} with ACE. "
        ).add_run(
            f"The 95% interval was entirely above zero for {', '.join(supported)}." if supported else
            "No model's 95% interval was entirely above zero."
        )
    doc.add_heading("Feature importance and SHAP", level=1)
    doc.add_paragraph(
        "Permutation importance measures the reduction in test ROC-AUC when a raw survey variable is "
        "shuffled, with 10 repeats. Error bars show the standard deviation across shuffles. Each raw category "
        "is shuffled as a unit before encoding. Orange denotes ACE features. Correlated variables can share "
        "predictive information, which can reduce individual permutation scores."
    )
    for name, group in importance.groupby("Model", sort=False):
        top = group.nlargest(3, "Mean ROC-AUC decrease")
        doc.add_paragraph(f"{name}: " + ", ".join(labels[f] for f in top.Feature) +
                          " had the largest mean permutation scores.")
    doc.add_picture(str(OUT / "permutation_importance.png"), width=Inches(6.4))
    top_shap = shap.nlargest(3, "Mean absolute SHAP")
    doc.add_paragraph(
        "TreeSHAP summarizes XGBoost contributions on the raw log-odds scale over all test rows. "
        "One-hot levels are summed within each original survey variable before averaging absolute values. "
        "The three largest mean absolute contributions were " + ", ".join(labels[f] for f in top_shap.Feature) + "."
    )
    doc.add_picture(str(OUT / "xgboost_shap.png"), width=Inches(5.8))
    doc.add_heading("Scope of inference", level=1)
    doc.add_paragraph(
        "This was an unweighted evaluation of a cross-sectional sample with complete SCD and ACE responses "
        "from 10 states. The results describe incremental classification of self-reported SCD in this holdout. "
        "Feature importances describe model behavior and support neither causal estimates nor estimates of future decline."
    )
    doc.save(OUT / "Part_D_Report_EN.docx")

## Fit and evaluate the six models

In [ ]:
def main():
    data, settings = load_inputs()
    train = data.Dataset_Split.eq("train")
    test = ~train
    x_train, x_test = data.loc[train], data.loc[test]
    y_train, y_test = x_train.SCD.astype(int), x_test.SCD.to_numpy(dtype=int)
    features = [col for col in data if col not in ("SCD", "Dataset_Split")]
    base = [col for col in features if not col.startswith("ACE")]
    assert (len(base), len(features), len(y_train), len(y_test), int(y_test.sum())) == (14, 25, 28603, 7151, 1223)
    pred = pd.DataFrame({"row_index": x_test.index, "y_true": y_test})
    metrics, importance, pairs = [], [], {}
    labels = feature_labels()
    fig_cm, cm_axes = plt.subplots(2, 3, figsize=(11, 7), layout="constrained")
    fig_roc, roc_axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")
    for column, name in enumerate(NAMES):
        pairs[name] = []
        for row, (feature_set, cols) in enumerate(zip(SETS, (base, features))):
            print(f"Evaluating {name} / {feature_set}", flush=True)
            model = make_pipeline(name, cols, settings[name, feature_set]).fit(x_train[cols], y_train)
            probs = model.predict_proba(x_test[cols])[:, 1]
            calls = (probs >= 0.5).astype(int)
            pairs[name].append((calls, probs))
            pred[f"{name} | {feature_set} | prediction"] = calls
            pred[f"{name} | {feature_set} | probability"] = probs
            matrix = confusion_matrix(y_test, calls, labels=[0, 1])
            metrics.append({
                "Model": name, "Features": feature_set,
                **{metric: score(y_test, probs if metric == "ROC-AUC" else calls)
                   for metric, score in METRICS.items()},
                **dict(zip(("TN", "FP", "FN", "TP"), matrix.ravel())),
                "Settings": repr(settings[name, feature_set]),
            })
            ax = cm_axes[row, column]
            ConfusionMatrixDisplay(matrix, display_labels=["No SCD", "SCD"]).plot(
                ax=ax, colorbar=False, cmap="Blues", im_kw={"vmin": 0, "vmax": 5928}
            )
            ax.set_title(f"{name}\n{feature_set}", fontsize=10)
            RocCurveDisplay.from_predictions(
                y_test, probs, ax=roc_axes[column], name=feature_set,
                color=("#536b87", "#d17838")[row],
            )
            if row == 1:
                scores = permutation_importance(model, x_test[cols], y_test, scoring="roc_auc",
                                                n_repeats=10, random_state=SEED)
                importance.extend({
                    "Model": name, "Feature": feature, "Mean ROC-AUC decrease": mean, "SD": sd,
                } for feature, mean, sd in zip(cols, scores.importances_mean, scores.importances_std))
                if name == "XGBoost":
                    shap = xgb_shap(model, x_test, cols)
        roc_axes[column].plot([0, 1], [0, 1], "--", color="0.65", linewidth=1)
        roc_axes[column].set_title(name, fontsize=10)
    fig_cm.suptitle("Holdout confusion matrices at a 0.5 probability cutoff")
    fig_roc.suptitle("Holdout ROC curves with and without ACE")
    save_figure(fig_cm, "confusion_matrices.png")
    save_figure(fig_roc, "roc_curves.png")

    print("Computing paired bootstrap intervals", flush=True)
    results = pd.DataFrame(metrics)
    changes = bootstrap_changes(y_test, pairs)
    deltas = []
    for name, group in results.groupby("Model", sort=False):
        baseline, expanded = group.iloc[0], group.iloc[1]
        lower, upper = changes[name]
        deltas.append({
            "Model": name, "F1 change": expanded.F1 - baseline.F1,
            "F1 95% CI": f"[{lower[0]:+.4f}, {upper[0]:+.4f}]",
            "AUC change": expanded["ROC-AUC"] - baseline["ROC-AUC"],
            "AUC 95% CI": f"[{lower[1]:+.4f}, {upper[1]:+.4f}]",
        })
    delta = pd.DataFrame(deltas)
    importance = pd.DataFrame(importance)
    plot_importance(importance, labels, "Mean ROC-AUC decrease", "SD", "ROC-AUC decrease",
                    "permutation_importance.png", "Top permutation importances for the three ACE models")
    plot_importance(shap, labels, "Mean absolute SHAP", None, "Mean absolute SHAP (log-odds)",
                    "xgboost_shap.png", "XGBoost with ACE: grouped TreeSHAP")
    for filename, frame in {
        "test_metrics.csv": results, "test_predictions.csv": pred, "ace_increment.csv": delta,
        "permutation_importance.csv": importance, "xgboost_shap.csv": shap,
    }.items():
        frame.to_csv(OUT / filename, index=False)

    original = pd.read_csv(ROOT / "analysisB" / "B_test_predictions.csv")
    np.testing.assert_array_equal(original.y_true, y_test)
    mismatch = [int(np.sum(original[f"{key}_pred"] != pairs[name][1][0]))
                for name, key in zip(NAMES[:2], ("logit", "rf"))]
    adherence = ("Relative to the archived Part B predictions, the refitted full logistic regression and "
                 f"random forest differed on {mismatch[0]} and {mismatch[1]} test classifications, respectively.")
    report(results, delta, importance, shap, labels, adherence)
    print(results.to_string(index=False))
    print(delta.to_string(index=False))
    print(adherence)


main()

## Review the outputs

In [ ]:
from IPython.display import Image, display

display(pd.read_csv(OUT / "test_metrics.csv").drop(columns="Settings"))
display(pd.read_csv(OUT / "ace_increment.csv"))
for filename in (
    "confusion_matrices.png", "roc_curves.png",
    "permutation_importance.png", "xgboost_shap.png",
):
    display(Image(filename=str(OUT / filename)))